# 1.DataSet Processing

In [ ]:
# ============================================================
# 1. DataSet Processing
# ============================================================
# Install the required NLP and ML libraries in the notebook environment.

!pip install -q sentence-transformers scikit-learn tqdm gdown transformers

import os
import json
import time
import random
import subprocess
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
# Define all local file paths used by the pipeline.
from tqdm.auto import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [ ]:
# ----------------------------
# File paths
# ----------------------------
from pathlib import Path

if "COLAB_GPU" in os.environ:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DATA_DIR = Path("/content/drive/MyDrive/climate-claim-verification/data")
else:
    # Local usage: notebook is inside notebooks/
    DATA_DIR = Path("../data")

TRAIN_FILE = DATA_DIR / "train-claims.json"
DEV_FILE = DATA_DIR / "dev-claims.json"
TEST_FILE = DATA_DIR / "test-claims-unlabelled.json"
EVIDENCE_FILE = DATA_DIR / "evidence.json"
EVAL_SCRIPT = DATA_DIR / "eval.py"

# EVIDENCE_GDRIVE_ID = "1JlUzRufknsHzKzvrEjgw8D3n_IRpjzo6"

# ----------------------------
# Reproducibility
# ----------------------------
RANDOM_SEED = 42
# The evidence file is large; this optional block downloads it only when it is missing.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ----------------------------
# Read a JSON file and return the parsed Python object.
# Download evidence.json if needed
# ----------------------------
# if not EVIDENCE_FILE.exists():
#     print("evidence.json not found. Downloading from Google Drive...")
#     !gdown --id 1JlUzRufknsHzKzvrEjgw8D3n_IRpjzo6 -O evidence.json
# else:
#     print("evidence.json already exists.")
# Load labelled train/dev claims, unlabelled test claims, and the full evidence corpus.

# ----------------------------
# Load data
# ----------------------------
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("\n--- Loading data ---")
train_claims  = load_json(TRAIN_FILE)
dev_claims    = load_json(DEV_FILE)
test_claims   = load_json(TEST_FILE)
evidence_dict = load_json(EVIDENCE_FILE)
# Retrieval hyperparameters selected from previous tuning experiments.

evidence_ids   = list(evidence_dict.keys())
evidence_texts = list(evidence_dict.values())

print(f"Train claims: {len(train_claims)}")
print(f"Dev claims:   {len(dev_claims)}")
print(f"Test claims:  {len(test_claims)}")
print(f"Evidence documents: {len(evidence_dict)}")

Mounted at /content/drive
Using device: cuda

--- Loading data ---
Train claims: 1228
Dev claims:   154
Test claims:  153
Evidence documents: 1208827


In [ ]:
# CrossEncoder training and inference settings for evidence reranking.
# ----------------------------
# Best retrieval config （from previous experiment ）
# ----------------------------
BEST_ALPHA = 0.03
BEST_GAP   = 0.03
BEST_MIN_K = 1
BEST_MAX_K = 6

TOP_K_A           = 250
TOP_K_B           = 250
FINAL_CANDIDATE_K = 400

# DeBERTa-based claim classifier settings.
NEGATIVE_PER_POSITIVE = 4
MAX_TRAIN_CLAIMS      = 3000

BASE_MODEL_NAME    = "cross-encoder/ms-marco-MiniLM-L-6-v2"
CE_MAX_LENGTH      = 512
TRAIN_BATCH_SIZE   = 16
PREDICT_BATCH_SIZE = 64
EPOCHS             = 3
LEARNING_RATE      = 1e-5
WARMUP_RATIO       = 0.1

# ----------------------------
# Map string labels to integer IDs required by the classifier loss function.
# Classification settings
# ----------------------------
CLF_MODEL_NAME          = "cross-encoder/nli-deberta-v3-small"
CLF_MAX_LENGTH          = 512
CLF_TRAIN_EPOCHS        = 5
# Retriever A uses unigram TF-IDF with English stopword removal for precise lexical matching.
CLF_LEARNING_RATE       = 2e-5
CLF_BATCH_SIZE          = 8
CLF_WARMUP_RATIO        = 0.1
CLF_MAX_EVIDENCE_TOKENS = 300

LABEL2ID = {"SUPPORTS": 0, "REFUTES": 1, "NOT_ENOUGH_INFO": 2, "DISPUTED": 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
print("\nLabel mapping:", LABEL2ID)


Label mapping: {'SUPPORTS': 0, 'REFUTES': 1, 'NOT_ENOUGH_INFO': 2, 'DISPUTED': 3}


In [ ]:
# ----------------------------
# TF-IDF Retriever A
# ----------------------------
print("\n--- Building TF-IDF retriever A ---")
# Retriever B uses unigram + bigram TF-IDF to capture short phrases and improve recall.
start_time = time.time()
vectorizer_a = TfidfVectorizer(
    token_pattern=r"(?u)\b[a-z0-9]+\b",
    lowercase=True,
    stop_words="english"
)
tfidf_matrix_a = vectorizer_a.fit_transform(evidence_texts)
print("TF-IDF A matrix shape:", tfidf_matrix_a.shape)
print(f"Build time: {time.time() - start_time:.2f}s")

# ----------------------------
# TF-IDF Retriever B
# ----------------------------
print("\n--- Building TF-IDF retriever B ---")
start_time = time.time()
vectorizer_b = TfidfVectorizer(
    token_pattern=r"(?u)\b[a-z0-9]+\b",
    lowercase=True,
# Return the indices of the highest-scoring items without fully sorting the whole evidence corpus.
    stop_words=None,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    max_features=500000
)
tfidf_matrix_b = vectorizer_b.fit_transform(evidence_texts)
# Scale scores to the 0-1 range so that scores from different retrievers can be combined.
print("TF-IDF B matrix shape:", tfidf_matrix_b.shape)
print(f"Build time: {time.time() - start_time:.2f}s")


--- Building TF-IDF retriever A ---
TF-IDF A matrix shape: (1208827, 531456)
Build time: 48.63s

--- Building TF-IDF retriever B ---
TF-IDF B matrix shape: (1208827, 500000)
Build time: 84.02s


In [ ]:
# ----------------------------
# Utility functions
# ----------------------------
def get_top_indices(scores, top_k):
    if top_k >= len(scores):
        return np.argsort(scores)[::-1]
# Run Retriever A for one claim and keep both raw and normalised scores for each candidate.
    top_indices = np.argpartition(scores, -top_k)[-top_k:]
    top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]
    return top_indices

def minmax_normalise(values):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return values
    v_min, v_max = float(np.min(values)), float(np.max(values))
    if abs(v_max - v_min) < 1e-12:
        return np.zeros_like(values)
    return (values - v_min) / (v_max - v_min)

def retrieve_a_with_scores(claim_text, top_k=TOP_K_A):
    claim_vec   = vectorizer_a.transform([claim_text])
    scores      = linear_kernel(claim_vec, tfidf_matrix_a).flatten()
    top_indices = get_top_indices(scores, top_k)
    raw_scores  = scores[top_indices]
    norm_scores = minmax_normalise(raw_scores)
# Run Retriever B for one claim using the broader unigram+bigram feature space.
    results = []
    for rank, (idx, raw_score, norm_score) in enumerate(
            zip(top_indices, raw_scores, norm_scores), start=1):
        results.append({
            "evidence_id":  evidence_ids[idx],
            "rank_a":       rank,
            "score_a":      float(raw_score),
            "norm_score_a": float(norm_score)
        })
    return results

def retrieve_b_with_scores(claim_text, top_k=TOP_K_B):
    claim_vec   = vectorizer_b.transform([claim_text])
    scores      = linear_kernel(claim_vec, tfidf_matrix_b).flatten()
    top_indices = get_top_indices(scores, top_k)
    raw_scores  = scores[top_indices]
    norm_scores = minmax_normalise(raw_scores)
    results = []
    for rank, (idx, raw_score, norm_score) in enumerate(
# Merge the two TF-IDF retrievers and attach ranking/score features used later for fusion.
            zip(top_indices, raw_scores, norm_scores), start=1):
        results.append({
            "evidence_id":  evidence_ids[idx],
            "rank_b":       rank,
            "score_b":      float(raw_score),
            "norm_score_b": float(norm_score)
        })
    return results

def retrieve_candidates_ensemble_with_features(
        claim_text, top_k_a=TOP_K_A, top_k_b=TOP_K_B, final_k=FINAL_CANDIDATE_K):
    results_a = retrieve_a_with_scores(claim_text, top_k_a)
    results_b = retrieve_b_with_scores(claim_text, top_k_b)
    candidate_info = {}
    for item in results_a:
        eid = item["evidence_id"]
        candidate_info[eid] = {
            "evidence_id":  eid, "rank_a": item["rank_a"], "rank_b": None,
            "score_a": item["score_a"], "score_b": 0.0,
            "norm_score_a": item["norm_score_a"], "norm_score_b": 0.0
        }
    for item in results_b:
# Reciprocal Rank Fusion rewards evidence that appears high in either retriever list.
        eid = item["evidence_id"]
        if eid not in candidate_info:
            candidate_info[eid] = {
                "evidence_id":  eid, "rank_a": None, "rank_b": item["rank_b"],
                "score_a": 0.0, "score_b": item["score_b"],
                "norm_score_a": 0.0, "norm_score_b": item["norm_score_b"]
            }
        else:
            candidate_info[eid]["rank_b"]       = item["rank_b"]
            candidate_info[eid]["score_b"]      = item["score_b"]
            candidate_info[eid]["norm_score_b"] = item["norm_score_b"]
# Interleave candidates from both retrievers to preserve diversity before cutting to final_k.
    for eid, info in candidate_info.items():
        rrf = 0.0
        if info["rank_a"] is not None:
            rrf += 1.0 / (60.0 + info["rank_a"])
        if info["rank_b"] is not None:
            rrf += 1.0 / (60.0 + info["rank_b"])
        info["rrf_score"]     = float(rrf)
        info["lexical_score"] = float(
            max(info["norm_score_a"], info["norm_score_b"]) + 5.0 * rrf)
    merged, seen = [], set()
    max_len = max(len(results_a), len(results_b))
    for i in range(max_len):
        if i < len(results_a):
            eid = results_a[i]["evidence_id"]
            if eid not in seen:
                seen.add(eid)
# Cache dev candidates once so later CrossEncoder scoring does not repeat TF-IDF retrieval.
                merged.append(candidate_info[eid])
        if i < len(results_b):
            eid = results_b[i]["evidence_id"]
            if eid not in seen:
                seen.add(eid)
                merged.append(candidate_info[eid])
        if len(merged) >= final_k:
            break
    return merged[:final_k]

In [ ]:
# ----------------------------
# Pre-compute dev AND train TF-IDF candidates
# Cache train candidates as well because classifier training uses retrieved evidence, not gold evidence.
# (train candidates needed later for classifier training)
# ----------------------------
print("\n--- Pre-computing dev TF-IDF candidates ---")
start_time = time.time()
dev_candidate_cache = {}
for claim_id, claim_info in tqdm(dev_claims.items(), desc="Dev TF-IDF retrieval"):
    claim_text = claim_info["claim_text"]
    dev_candidate_cache[claim_id] = {
        "claim_text":      claim_text,
        "candidate_infos": retrieve_candidates_ensemble_with_features(claim_text)
    }
print(f"Done in {time.time() - start_time:.2f}s")

print("\n--- Pre-computing train TF-IDF candidates ---")
start_time = time.time()
train_candidate_cache = {}
for claim_id, claim_info in tqdm(train_claims.items(), desc="Train TF-IDF retrieval"):
    claim_text = claim_info["claim_text"]
    train_candidate_cache[claim_id] = {
        "claim_text":      claim_text,
        "candidate_infos": retrieve_candidates_ensemble_with_features(claim_text)
    }
print(f"Done in {time.time() - start_time:.2f}s")


--- Pre-computing dev TF-IDF candidates ---


Dev TF-IDF retrieval:   0%|          | 0/154 [00:00<?, ?it/s]

Done in 242.19s

--- Pre-computing train TF-IDF candidates ---


Train TF-IDF retrieval:   0%|          | 0/1228 [00:00<?, ?it/s]

Done in 1859.53s


# 2. Model Implementation

## Pipeline Overview

This section implements a **two-stage retrieve-then-classify** pipeline:

1. **Dual TF-IDF Ensemble Retrieval** — Two TF-IDF retrievers (unigram and unigram+bigram) independently retrieve top-250 candidates each and are fused via **Reciprocal Rank Fusion (RRF)**, producing a final candidate pool of up to 400 passages.

2. **CrossEncoder Reranking** (`ms-marco-MiniLM-L-6-v2`) — Fine-tuned on binary relevance pairs (gold evidence as positive, random samples as negative). Scores every candidate against the claim with full cross-attention, then fuses semantic score with the lexical RRF score (α=0.03). A gap-based selection strategy adaptively picks 1–6 evidence passages per claim.

3. **DeBERTa-v3 Claim Classifier** (`nli-deberta-v3-small`) — Fine-tuned for 4-class claim verification (SUPPORTS / REFUTES / NOT_ENOUGH_INFO / DISPUTED). Trained on CrossEncoder-retrieved evidence (not gold) to match the inference distribution. Sqrt-weighted cross-entropy mitigates class imbalance.

In [ ]:
# ============================================================
# 2. Model Implementation
# ============================================================

from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW

# ============================================================
# Part A: Retrieval CrossEncoder
# ============================================================

# Build binary training pairs for the CrossEncoder: gold evidence is positive, sampled evidence is negative.
print("\n--- Building CrossEncoder training pairs ---")
start_time = time.time()

# Each InputExample contains a claim-evidence pair and a relevance label.
train_samples    = []
train_items      = list(train_claims.items())
if MAX_TRAIN_CLAIMS is not None:
    train_items = train_items[:MAX_TRAIN_CLAIMS]
all_evidence_set = set(evidence_ids)

# Iterate through each training claim and create positive/negative evidence pairs.
for claim_id, claim_info in tqdm(train_items, desc="Pair construction"):
    claim_text        = claim_info["claim_text"]
    gold_evidence_ids = claim_info.get("evidences", [])
    gold_set          = set(gold_evidence_ids)
# Positive examples teach the model what truly relevant evidence looks like.
    for gold_id in gold_evidence_ids:
        if gold_id in evidence_dict:
            train_samples.append(
                InputExample(texts=[claim_text, evidence_dict[gold_id]], label=1.0))
# Negative examples are sampled from non-gold evidence to teach the model to reject irrelevant matches.
    n_negatives = len(gold_evidence_ids) * NEGATIVE_PER_POSITIVE
    if n_negatives <= 0:
        continue
    negative_pool = list(all_evidence_set - gold_set)
    for neg_id in random.sample(negative_pool, min(n_negatives, len(negative_pool))):
        train_samples.append(
            InputExample(texts=[claim_text, evidence_dict[neg_id]], label=0.0))

print(f"Training samples: {len(train_samples)}")
print(f"Time: {(time.time() - start_time) / 60:.2f} min")

# Fine-tune the CrossEncoder reranker on the constructed relevance pairs.
print("\n--- Fine-tuning CrossEncoder ---")
ce_model = CrossEncoder(BASE_MODEL_NAME, num_labels=1, max_length=CE_MAX_LENGTH, device=DEVICE)
train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=TRAIN_BATCH_SIZE)
warmup_steps = int(len(train_dataloader) * EPOCHS * WARMUP_RATIO)

start_time = time.time()
ce_model.fit(
    train_dataloader=train_dataloader,
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": LEARNING_RATE},
    show_progress_bar=True
)
print(f"CrossEncoder fine-tuning time: {(time.time() - start_time) / 60:.2f} min")

# ----------------------------
# Scoring / fusion / selection helpers
# ----------------------------
# Score all candidate evidence items for one claim using the fine-tuned CrossEncoder.
def score_claim_candidates_with_features(claim_text, candidate_infos):
    pairs = [[claim_text, evidence_dict.get(item["evidence_id"], "")]
             for item in candidate_infos]
    ce_scores = ce_model.predict(pairs, batch_size=PREDICT_BATCH_SIZE, show_progress_bar=False)
    scored_items = []
    for item, ce_score in zip(candidate_infos, ce_scores):
        new_item = dict(item)
        new_item["ce_score"] = float(ce_score)
        scored_items.append(new_item)
    scored_items.sort(key=lambda x: x["ce_score"], reverse=True)
    return scored_items

# Combine semantic CrossEncoder scores with a small lexical score contribution.
def rank_with_score_fusion(scored_items, alpha=BEST_ALPHA):
    if len(scored_items) == 0:
        return []
    ce_values  = np.array([x["ce_score"]     for x in scored_items], dtype=float)
    lex_values = np.array([x["lexical_score"] for x in scored_items], dtype=float)
# Normalise scores before fusion because CE and TF-IDF scores are on different scales.
    norm_ce    = minmax_normalise(ce_values)
    norm_lex   = minmax_normalise(lex_values)
    fused = []
    for item, cn, ln in zip(scored_items, norm_ce, norm_lex):
        new_item = dict(item)
        new_item["norm_ce_score"]      = float(cn)
        new_item["norm_lexical_score"] = float(ln)
# alpha controls how much lexical TF-IDF/RRF evidence contributes to the final ranking.
        new_item["final_score"]        = float((1.0 - alpha) * cn + alpha * ln)
        fused.append(new_item)
    fused.sort(key=lambda x: x["final_score"], reverse=True)
    return fused

# Select a variable number of evidences based on how close their scores are to the top item.
def select_evidences_by_gap(fused_items, gap=BEST_GAP, min_k=BEST_MIN_K, max_k=BEST_MAX_K):
    if len(fused_items) == 0:
        return []
    top_score = fused_items[0]["final_score"]
    selected  = [x["evidence_id"] for x in fused_items
                 if (top_score - x["final_score"]) <= gap]
    if len(selected) < min_k:
        selected = [x["evidence_id"] for x in fused_items[:min_k]]
    return selected[:max_k]

# ----------------------------
# Score dev candidates with CrossEncoder
# ----------------------------
# Apply the CrossEncoder to all dev candidates so retrieval quality can be evaluated.
print("\n--- Scoring dev candidates with CrossEncoder ---")
start_time = time.time()
dev_scored_cache = {}
for i, (claim_id, info) in enumerate(dev_candidate_cache.items(), start=1):
    dev_scored_cache[claim_id] = {
        "claim_text":   info["claim_text"],
        "scored_items": score_claim_candidates_with_features(
            info["claim_text"], info["candidate_infos"])
    }
    if i % 10 == 0 or i == len(dev_candidate_cache):
        elapsed = time.time() - start_time
        print(f"[Dev] {i}/{len(dev_candidate_cache)} | {elapsed/60:.1f}min | {elapsed/i:.2f}s/claim")
print(f"Dev scoring time: {(time.time() - start_time) / 60:.2f} min")

# ----------------------------
# Score train candidates with CrossEncoder
# classifier trains on the SAME evidence distribution as inference
# ----------------------------
# Score train candidates too; the classifier should train on the same retrieved-evidence distribution used at inference.
print("\n--- Scoring train candidates with CrossEncoder ---")
start_time = time.time()
train_scored_cache = {}
for i, (claim_id, info) in enumerate(train_candidate_cache.items(), start=1):
    train_scored_cache[claim_id] = {
        "claim_text":   info["claim_text"],
        "scored_items": score_claim_candidates_with_features(
            info["claim_text"], info["candidate_infos"])
    }
    if i % 50 == 0 or i == len(train_candidate_cache):
        elapsed = time.time() - start_time
        print(f"[Train] {i}/{len(train_candidate_cache)} | {elapsed/60:.1f}min | {elapsed/i:.2f}s/claim")
print(f"Train scoring time: {(time.time() - start_time) / 60:.2f} min")


# ============================================================
# Part B: Claim Classification
#
# Changes vs baseline:
#   1. Train on CrossEncoder-retrieved evidence (not gold)
#      → training distribution matches inference exactly
#   2. Sqrt-weighted cross-entropy loss
#      → fixes class imbalance without over-correcting
#   3. Best-checkpoint saving on retrieved-evidence dev_acc
#      → correct early-stopping signal
# ============================================================

# Concatenate selected evidence texts into one classifier context, separated by [SEP].
def build_evidence_context(evidence_id_list):
    parts  = [evidence_dict.get(eid, "") for eid in evidence_id_list]
    joined = " [SEP] ".join(parts)
    words  = joined.split()
    if len(words) > CLF_MAX_EVIDENCE_TOKENS:
        joined = " ".join(words[:CLF_MAX_EVIDENCE_TOKENS])
    return joined

# Minimal PyTorch Dataset wrapper for claim classification examples.
class ClaimDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        return self.samples[idx]

# ----------------------------
# Build train samples from CrossEncoder-retrieved evidence
# ----------------------------
# Build classifier training examples from retrieved evidence instead of gold evidence.
def build_clf_train_samples():
    samples = []
    for claim_id, info in train_scored_cache.items():
        label_str  = train_claims[claim_id].get("claim_label", "NOT_ENOUGH_INFO")
        label_id   = LABEL2ID.get(label_str, 2)
        fused      = rank_with_score_fusion(info["scored_items"])
        ret_eids   = select_evidences_by_gap(fused)
        ev_context = build_evidence_context(ret_eids)
        samples.append((info["claim_text"], ev_context, label_id))
    return samples

# Dev samples also use retrieved evidence (consistent with inference)
# Build dev examples the same way as inference, using retrieved evidence.
def build_clf_dev_samples():
    samples = []
    for claim_id, info in dev_scored_cache.items():
        label_str  = dev_claims[claim_id].get("claim_label")
        label_id   = LABEL2ID.get(label_str, 2)
        fused      = rank_with_score_fusion(info["scored_items"])
        ret_eids   = select_evidences_by_gap(fused)
        ev_context = build_evidence_context(ret_eids)
        samples.append((info["claim_text"], ev_context, label_id))
    return samples

# ----------------------------
# Tokeniser + collate
# ----------------------------
# Tokeniser encodes claim and evidence context as a sentence-pair input.
clf_tokenizer = AutoTokenizer.from_pretrained(CLF_MODEL_NAME)

# Batch collation pads/truncates claim-evidence pairs and attaches labels.
def clf_collate(batch):
    claims    = [x[0] for x in batch]
    evidences = [x[1] for x in batch]
    labels    = torch.tensor([x[2] for x in batch], dtype=torch.long)
    encoding  = clf_tokenizer(
        claims, evidences,
        padding=True, truncation=True,
        max_length=CLF_MAX_LENGTH, return_tensors="pt"
    )
    encoding["labels"] = labels
    return encoding

# ----------------------------
# Load model
# ----------------------------
# Load a pretrained sequence classifier and replace/resize the output layer for four labels.
print("\n--- Loading classifier model:", CLF_MODEL_NAME, "---")
clf_model = AutoModelForSequenceClassification.from_pretrained(
    CLF_MODEL_NAME, num_labels=4, ignore_mismatched_sizes=True)
clf_model = clf_model.to(DEVICE)

# ----------------------------
# Build datasets
# ----------------------------
# Materialise train/dev examples after retrieval and reranking are complete.
print("\n--- Building classifier samples from CrossEncoder-retrieved evidence ---")
clf_train_samples = build_clf_train_samples()
clf_dev_samples   = build_clf_dev_samples()

print(f"Train samples: {len(clf_train_samples)}")
print(f"Dev samples:   {len(clf_dev_samples)}")

train_label_dist = Counter(s[2] for s in clf_train_samples)
print("Train label distribution:", {ID2LABEL[k]: v for k, v in sorted(train_label_dist.items())})

clf_train_loader = DataLoader(
    ClaimDataset(clf_train_samples), batch_size=CLF_BATCH_SIZE,
    shuffle=True, collate_fn=clf_collate)
clf_dev_loader = DataLoader(
    ClaimDataset(clf_dev_samples), batch_size=CLF_BATCH_SIZE,
    shuffle=False, collate_fn=clf_collate)

# Compute class weights to reduce bias towards majority labels.
# ----------------------------
# Sqrt-weighted cross-entropy loss
# Mild correction for class imbalance — avoids over-correcting
# ----------------------------
total_train  = sum(train_label_dist.values())
class_weights = torch.tensor(
    [np.sqrt(total_train / (4 * max(train_label_dist[i], 1))) for i in range(4)],
    dtype=torch.float
).to(DEVICE)

print("\nClass weights (sqrt, mild correction):")
for i in range(4):
    print(f"  {ID2LABEL[i]:<18}: {class_weights[i].item():.3f}")

# Weighted cross-entropy makes errors on rare classes count slightly more.
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

# ----------------------------
# Optimiser + scheduler
# ----------------------------
# Use AdamW with a linear warmup schedule for stable fine-tuning.
total_steps      = len(clf_train_loader) * CLF_TRAIN_EPOCHS
warmup_steps_clf = int(total_steps * CLF_WARMUP_RATIO)
optimizer_clf    = AdamW(clf_model.parameters(), lr=CLF_LEARNING_RATE)
scheduler_clf    = get_linear_schedule_with_warmup(
    optimizer_clf, num_warmup_steps=warmup_steps_clf, num_training_steps=total_steps)

# ----------------------------
#  Best checkpoint on retrieved-evidence dev_acc
# ----------------------------
# Track the best checkpoint based on dev accuracy using retrieved evidence.
best_dev_acc = 0.0
best_state   = None

print("\n--- Fine-tuning classifier ---")
print(f"Model: {CLF_MODEL_NAME}")
print(f"Epochs: {CLF_TRAIN_EPOCHS}, LR: {CLF_LEARNING_RATE}, Batch: {CLF_BATCH_SIZE}")
print("Early stopping on: dev_acc with RETRIEVED evidence\n")

start_time = time.time()

# Main classifier fine-tuning loop.
for epoch in range(1, CLF_TRAIN_EPOCHS + 1):
    # train
    clf_model.train()
    total_loss, total_correct, total_samples = 0.0, 0, 0
# Training step: forward pass, weighted loss, gradient clipping, optimiser and scheduler updates.
    for batch in clf_train_loader:
        labels = batch.pop("labels").to(DEVICE)
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = clf_model(**batch)
        loss    = loss_fn(outputs.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(clf_model.parameters(), 1.0)
        optimizer_clf.step()
        scheduler_clf.step()
        optimizer_clf.zero_grad()
        total_loss    += loss.item() * labels.size(0)
        total_correct += (outputs.logits.argmax(-1) == labels).sum().item()
        total_samples += labels.size(0)

    train_loss = total_loss / total_samples
    train_acc  = total_correct / total_samples

# Evaluate on dev data after each epoch without updating model weights.
    # dev eval on retrieved evidence
    clf_model.eval()
    dev_correct, dev_total = 0, 0
    with torch.no_grad():
        for batch in clf_dev_loader:
            labels = batch.pop("labels").to(DEVICE)
            batch  = {k: v.to(DEVICE) for k, v in batch.items()}
            preds  = clf_model(**batch).logits.argmax(-1)
            dev_correct += (preds == labels).sum().item()
            dev_total   += labels.size(0)

    dev_acc = dev_correct / dev_total
    elapsed = (time.time() - start_time) / 60

# Save the best-performing model state in memory for later restoration.
    marker = ""
    if dev_acc > best_dev_acc:
        best_dev_acc = dev_acc
        best_state   = {k: v.cpu().clone() for k, v in clf_model.state_dict().items()}
        marker       = "  *** best ***"

    print(f"Epoch {epoch}/{CLF_TRAIN_EPOCHS} | loss={train_loss:.4f} | "
          f"train_acc={train_acc:.4f} | dev_acc(retrieved)={dev_acc:.4f} | "
          f"elapsed={elapsed:.1f}min{marker}")

# Restore the checkpoint that performed best on dev retrieved evidence.
print(f"\nRestoring best checkpoint  (dev_acc={best_dev_acc:.4f})")
clf_model.load_state_dict(best_state)
clf_model.to(DEVICE)
clf_model.eval()
print(f"Total classifier time: {(time.time() - start_time) / 60:.2f} min")

# ----------------------------
# Inference
# ----------------------------
# Predict the final claim label for a claim and its selected evidence IDs.
def classify_claim(claim_text, retrieved_evidence_ids):
    ev_context = build_evidence_context(retrieved_evidence_ids)
    encoding   = clf_tokenizer(
        claim_text, ev_context,
        truncation=True, max_length=CLF_MAX_LENGTH, return_tensors="pt")
    encoding = {k: v.to(DEVICE) for k, v in encoding.items()}
    with torch.no_grad():
        logits = clf_model(**encoding).logits
    return ID2LABEL[logits.argmax(-1).item()]



--- Building CrossEncoder training pairs ---


Pair construction:   0%|          | 0/1228 [00:00<?, ?it/s]

Training samples: 20610
Time: 4.46 min

--- Fine-tuning CrossEncoder ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

...


--- Loading classifier model: cross-encoder/nli-deberta-v3-small ---


model.safetensors:   0%|          | 0.00/568M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([4, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([4])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.



--- Building classifier samples from CrossEncoder-retrieved evidence ---
Train samples: 1228
Dev samples:   154
Train label distribution: {'SUPPORTS': 519, 'REFUTES': 199, 'NOT_ENOUGH_INFO': 386, 'DISPUTED': 124}

Class weights (sqrt, mild correction):
  SUPPORTS          : 0.769
  REFUTES           : 1.242
  NOT_ENOUGH_INFO   : 0.892
  DISPUTED          : 1.573

--- Fine-tuning classifier ---
Model: cross-encoder/nli-deberta-v3-small
Epochs: 5, LR: 2e-05, Batch: 8
Early stopping on: dev_acc with RETRIEVED evidence

Epoch 1/5 | loss=1.3423 | train_acc=0.3982 | dev_acc(retrieved)=0.4870 | elapsed=1.0min  *** best ***
Epoch 2/5 | loss=1.1785 | train_acc=0.5252 | dev_acc(retrieved)=0.5325 | elapsed=1.9min  *** best ***
Epoch 3/5 | loss=0.9836 | train_acc=0.6270 | dev_acc(retrieved)=0.5649 | elapsed=2.9min  *** best ***
Epoch 4/5 | loss=0.8017 | train_acc=0.7264 | dev_acc(retrieved)=0.5390 | elapsed=3.8min
Epoch 5/5 | loss=0.6784 | train_acc=0.7736 | dev_acc(retrieved)=0.5325 | elapsed=4.

# 3. Testing and Evaluation

Evaluation uses the official `eval.py` script. The combined metric is the **harmonic mean** of Evidence Retrieval F-score and Claim Classification Accuracy, ensuring both components must perform well for a high overall score.

This section: generates dev predictions → runs official evaluator → generates test output → sanity checks.

In [13]:
import os
print(os.path.exists(EVAL_SCRIPT))

True


In [14]:
# ============================================================
# 3. Testing and Evaluation
# ============================================================

# Save prediction dictionaries in the JSON format expected by the evaluator/Codabench.
def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)

# Run the official evaluation script and capture its printed metrics.
def run_eval(prediction_file, gold_file=DEV_FILE):
    result = subprocess.run(
        ["python", str(EVAL_SCRIPT), "--predictions", str(prediction_file),
         "--groundtruth", str(gold_file)],
        capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result.stdout

# Extract F-score, accuracy, and harmonic mean from the evaluator output text.
def parse_eval_output(output_text):
    f_score, acc, hmean = None, None, None
    for line in output_text.splitlines():
        if "Evidence Retrieval F-score" in line:
            f_score = float(line.split("=")[-1].strip())
        elif "Claim Classification Accuracy" in line:
            acc = float(line.split("=")[-1].strip())
        elif "Harmonic Mean" in line:
            hmean = float(line.split("=")[-1].strip())
    return f_score, acc, hmean

# Convert a scored cache into final predictions by selecting evidence and classifying each claim.
def build_predictions(scored_cache):
    predictions = {}
    for claim_id, info in scored_cache.items():
        fused_items           = rank_with_score_fusion(info["scored_items"])
        evidence_ids_selected = select_evidences_by_gap(fused_items)
        predicted_label       = classify_claim(info["claim_text"], evidence_ids_selected)
        predictions[claim_id] = {
            "claim_text":  info["claim_text"],
            "claim_label": predicted_label,
            "evidences":   evidence_ids_selected
        }
    return predictions

# ----------------------------
# Dev evaluation
# ----------------------------
print("\n" + "=" * 70)
print("DEV EVALUATION")
print("=" * 70)

# Build and save dev predictions first so the model can be checked before generating test output.
dev_predictions = build_predictions(dev_scored_cache)
dev_pred_file   = "dev-best.json"
save_json(dev_predictions, dev_pred_file)

# Evaluate dev predictions against the labelled dev set.
output = run_eval(dev_pred_file, DEV_FILE)
f_score, acc, hmean = parse_eval_output(output)

print(f"F={f_score:.6f}, A={acc:.6f}, Mean={hmean:.6f}")

# Compare predicted and gold label distributions to diagnose classifier bias.
pred_labels = [v["claim_label"] for v in dev_predictions.values()]
gold_labels = [v["claim_label"] for v in dev_claims.values()]
print("\nPredicted label distribution:", dict(Counter(pred_labels)))
print("Gold label distribution:     ", dict(Counter(gold_labels)))

# ----------------------------
# Test predictions
# ----------------------------
print("\n" + "=" * 70)
print("GENERATING TEST PREDICTIONS")
print("=" * 70)

# Generate final predictions for the unlabelled test set using the full retrieval-rerank-classify pipeline.
start_time       = time.time()
test_predictions = {}

# Process each test claim independently because test candidates are not pre-cached.
for i, (claim_id, claim_info) in enumerate(test_claims.items(), start=1):
    claim_text      = claim_info["claim_text"]
    candidate_infos = retrieve_candidates_ensemble_with_features(claim_text)
    scored_items    = score_claim_candidates_with_features(claim_text, candidate_infos)
    fused_items     = rank_with_score_fusion(scored_items)
    ev_selected     = select_evidences_by_gap(fused_items)
    pred_label      = classify_claim(claim_text, ev_selected)

    test_predictions[claim_id] = {
        "claim_text":  claim_text,
        "claim_label": pred_label,
        "evidences":   ev_selected
    }

# Print progress because CrossEncoder scoring over all test candidates can be slow.
    if i % 10 == 0 or i == len(test_claims):
        elapsed = time.time() - start_time
        print(f"[Progress] {i}/{len(test_claims)} | "
              f"elapsed={elapsed/60:.1f}min | avg={elapsed/i:.2f}s/claim")

# Save the file that should be submitted after being zipped.
output_file = "test-output.json"
save_json(test_predictions, output_file)
print("\nSaved:", output_file)

# ----------------------------
# Sanity check
# ----------------------------
print("\n" + "=" * 70)
print("SANITY CHECK")
print("=" * 70)

# Reload the saved output and check basic structural validity.
preds = load_json(output_file)
print("Test claims:   ", len(test_claims))
print("Predictions:   ", len(preds))
print("Missing:       ", len(set(test_claims.keys()) - set(preds.keys())))

label_counts        = {}
evidence_len_counts = {}
invalid_eids        = []

# Check label counts, evidence-count distribution, and invalid evidence IDs.
for claim_id, pred in preds.items():
    label     = pred.get("claim_label")
    evidences = pred.get("evidences", [])
    label_counts[label]                 = label_counts.get(label, 0) + 1
    evidence_len_counts[len(evidences)] = evidence_len_counts.get(len(evidences), 0) + 1
    for eid in evidences:
        if eid not in evidence_dict:
            invalid_eids.append((claim_id, eid))

print("\nLabel distribution:", label_counts)
print("Evidence count distribution:", evidence_len_counts)
print("Invalid evidence IDs:", len(invalid_eids))

!zip -q submission.zip test-output.json
print("\nCreated submission.zip")



DEV EVALUATION
Evidence Retrieval F-score (F)    = 0.24197634976855756
Claim Classification Accuracy (A) = 0.564935064935065
Harmonic Mean of F and A          = 0.33882511110456937

F=0.241976, A=0.564935, Mean=0.338825

Predicted label distribution: {'SUPPORTS': 80, 'REFUTES': 39, 'NOT_ENOUGH_INFO': 30, 'DISPUTED': 5}
Gold label distribution:      {'SUPPORTS': 68, 'NOT_ENOUGH_INFO': 41, 'REFUTES': 27, 'DISPUTED': 18}

GENERATING TEST PREDICTIONS
[Progress] 10/153 | elapsed=0.6min | avg=3.57s/claim
[Progress] 20/153 | elapsed=1.2min | avg=3.57s/claim
[Progress] 30/153 | elapsed=1.7min | avg=3.48s/claim
[Progress] 40/153 | elapsed=2.4min | avg=3.53s/claim
[Progress] 50/153 | elapsed=3.0min | avg=3.56s/claim
[Progress] 60/153 | elapsed=3.5min | avg=3.52s/claim
[Progress] 70/153 | elapsed=4.1min | avg=3.54s/claim
[Progress] 80/153 | elapsed=4.7min | avg=3.52s/claim
[Progress] 90/153 | elapsed=5.3min | avg=3.51s/claim
[Progress] 100/153 | elapsed=5.8min | avg=3.50s/claim
[Progress] 110/15